<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_22_wideband_widefield_algorithm_boundaries.ipynb">9.22 宽带宽场算法边界</a></li>
        <li>下一节： <a href="9_24_software_ecosystem_reproducibility.ipynb">9.24 软件生态与可复现实践</a></li>
    </ul>
</div>


## 9.23 观测设计与归档数据再分析：从科学目标到可复现结果

前面的实践章节主要从已有数据出发，讨论检查、校准、成像、自校准、谱线、偏振、短间距和宽带宽场处理。本节把视角向前和向后各延伸一步：向前看，科学目标如何转化为观测设置；向后看，归档数据是否仍能支撑新的科学问题。真实研究中，许多处理困难并不是从 CASA 或成像器参数开始的，而是在观测设计阶段已经被写入数据：频率不覆盖目标谱线，通道宽度过粗，阵列配置缺少大尺度结构，校准源间隔过长，或归档数据的权重和 flag 历史无法追溯。

观测设计不是行政性的表格填写，而是测量方程、噪声、校准和科学量之间的闭合。一个好的设计能让后续处理有足够约束；一个不匹配的设计即使经过复杂成像，也只能在有限信息中补救。归档数据再分析同样如此：数据已经存在并不意味着它适合回答新的问题。必须先判断它的频率、uv 覆盖、校准质量、噪声、主波束位置和 pipeline 假设是否与新目标一致。


### 9.23.1 从科学问题到观测需求

科学问题首先要被翻译成可观测量。若目标是测量点源位置，关键量是角分辨率、信噪比和相位参考稳定性；若目标是弥散星系盘的总 H I 质量，关键量是短间距、表面亮度灵敏度、速度覆盖和通量尺度；若目标是偏振 RM 图，关键量是 $\lambda^2$ 覆盖、偏振校准、RMSF 和离轴 leakage；若目标是变源监测，关键量是时间采样、校准一致性和绝对通量稳定性。

这一步应尽量把“希望看到什么”改写成数值要求：角分辨率需要多少，最大角尺度有多大，目标线宽和系统速度范围是多少，期望通量或亮温是多少，动态范围需要到什么量级，是否需要偏振角绝对校准，是否允许依赖 pipeline 图像产品。只有这些量明确，频率、带宽、通道宽度、积分时间、阵列配置和校准源选择才有物理依据。


![observing requirements flow](figures/practical_observing_requirements_flow.png)

图中把观测设计写成一条从科学问题到可复现结果的链。科学目标决定可观测量，可观测量决定观测设置，观测设置决定校准计划和数据产品，最后影响误差预算和结果报告。这个链条也适用于归档数据再分析：若某个中间环节与新问题不匹配，后续处理只能说明限制，而不能把缺失的信息重新制造出来。


### 9.23.2 阵列配置：角分辨率、最大可恢复尺度和表面亮度灵敏度

阵列配置最常被简化为“分辨率选择”，但它同时决定至少三个量。最高空间频率近似给出合成波束：

$$
\theta_{\rm res}\sim \frac{\lambda}{b_{\rm max}}.
$$

最低空间频率给出最大可恢复尺度的量级：

$$
\theta_{\rm MRS}\sim \kappa\frac{\lambda}{b_{\rm min}}.
$$

热噪声以 $\mathrm{Jy\,beam^{-1}}$ 表示时，高分辨率图像看起来可能很灵敏；但若目标是扩展结构的亮温或表面亮度，高分辨率会把同一总通量分散到更多 beam 中，使每 beam 信噪比下降。因此，阵列配置必须同时匹配目标最小结构和最大结构。单纯追求最高分辨率，常会牺牲低面亮度发射和总通量恢复。


![resolution MRS sensitivity](figures/practical_observing_resolution_mrs_sensitivity_tradeoff.png)

左图示意不同配置下合成波束和最大可恢复尺度是两个不同尺度。右图示意扩展源被越来越小的 beam 分割后，表面亮度信噪比会下降。真实设计中常需要多配置组合，或在高分辨率和低面亮度灵敏度之间选择科学优先级。


### 9.23.3 频率、带宽和通道宽度

频率选择首先由科学目标决定：连续谱研究关心目标辐射机制、谱指数、RFI 环境和主波束大小；谱线研究还要满足静止频率、红移或系统速度、速度覆盖和速度分辨率。谱线通道宽度与速度宽度近似满足

$$
\Delta v \simeq c\frac{\Delta\nu}{\nu_0}.
$$

通道太宽会稀释窄线、影响线中心和线宽；通道太窄会增加数据量，并在后续成像中提高计算成本。若需要连续谱扣除，频带还必须包含足够 line-free 通道。若目标速度不确定，还需要留出速度覆盖余量，而不是只覆盖理论中心频率附近的一小段。

连续谱宽带设计也有权衡。更宽带宽提高热噪声灵敏度，并提供谱指数信息；但更宽频带会增强频变主波束、频变 uv 覆盖、带宽 smearing、RFI 和 spectral curvature 的影响。后续是否使用 MT-MFS、wideband PB correction 或按子带成像，往往在观测设计阶段就已经被频带选择决定。


![spectral setup](figures/practical_observing_spectral_setup_channel_width.png)

左图说明不同通道宽度对线型信息的影响。窄通道保留速度结构，但噪声和数据量更大；宽通道提高每通道信噪比，却会抹平窄线和线翼。右图把连续谱带宽、谱线窗口和后续 averaging 放在同一条链中。频谱设置不是只为了“覆盖目标频率”，还要支持校准、连续谱扣除、RFI excision 和后续科学测量。


### 9.23.4 积分时间、校准节奏和观测效率

热噪声近似随带宽和积分时间下降：

$$
\sigma_S \propto \frac{\mathrm{SEFD}}{\sqrt{N_{\rm pol}\,N_{\rm ant}(N_{\rm ant}-1)\,\Delta\nu\,t}}.
$$

这个公式说明带宽、天线数和总积分时间的重要性，但它只描述热噪声。高动态范围成像、偏振、长基线或高频观测常由校准误差限制。相位校准源间隔过长会导致相位转移不稳定；bandpass 校准不足会留下频率结构；通量校准源选择不当会影响绝对通量尺度；偏振观测还需要合适的 D-term、交叉手相位和绝对角度校准。

校准开销不是浪费时间，而是把系统误差控制在科学目标允许范围内。低频观测可能需要更强的 RFI 策略和方向相关校准；高频观测常需要更快的 phase referencing、指向校正和天气筛选。若只按热噪声计算积分时间，而没有给校准和天气留余量，最终图像可能远达不到理论灵敏度。


![calibration schedule](figures/practical_observing_calibration_schedule_overheads.png)

图中用时间轴示意通量、带通、相位校准和目标扫描的关系。相位不稳定越强，校准循环越需要密集；但校准越密集，目标 on-source 时间越少。观测设计的任务是在热噪声、系统误差和观测效率之间取得平衡。


### 9.23.5 归档数据：先判断能不能回答问题

归档数据再分析的第一步不是下载数据并直接成像，而是判断数据是否适合新的科学问题。需要检查项目原始目标、频率设置、阵列配置、观测日期、校准源、天气或电离层状况、RFI 环境、flag 比例、pipeline 版本、权重定义和已有图像产品。一个为点源连续谱监测设计的数据集，未必适合测量弥散低面亮度结构；一个为总强度设计的数据集，未必具备可靠偏振校准；一个 pipeline 图像看起来漂亮，也未必在主波束边缘或低信噪比谱线区域满足新的测量要求。

归档数据的 QA 报告和 weblog 很关键。它们通常记录 calibrator 解的稳定性、flag 摘要、天气或系统温度、bandpass 质量、成像参数、残差图和 pipeline 判断。再分析时应特别关注权重是否被重标定、哪些数据被 flag、是否存在多个 execution block、不同配置或不同日期之间通量尺度是否一致，以及 pipeline mask 是否过浅或过深。没有 provenance 的结果很难复现，也很难解释与原 pipeline 产品的差异。


![archive QA decision tree](figures/practical_archive_reanalysis_qa_decision_tree.png)

图中给出归档数据再分析的判断链。先确认科学目标与观测设置是否匹配，再读 metadata、QA 和 weblog；若 pipeline 产品已经满足科学区域、噪声、beam 和通量要求，可以直接测量并记录版本；若只是成像参数不合适，可以重做 flag、权重或 imaging；若校准模型、flag 历史或数据权重不满足要求，则需要完整重处理，甚至判断该归档数据不适合当前问题。


### 9.23.6 结果报告中的观测与归档信息

一个基于新观测或归档数据的结果，至少应报告：项目编号和观测日期；目标频率、带宽、通道宽度和速度约定；阵列配置、合成波束、最大可恢复尺度估计和 on-source 时间；校准源和校准策略；flag 比例和关键 QA 结论；成像权重、像元、图像大小、主波束处理和 deconvolution 设置；若使用 pipeline 产品，应报告 pipeline 版本、产品类型和是否重新处理；若使用归档数据，应说明原项目目标与当前科学目标的差异。

这些信息并不是论文附属细节，而是判断结果可比性和可复现性的基础。同一个通量差异可能来自真实变源，也可能来自不同阵列配置、主波束校正、缺短间距或权重策略。同一个谱线线宽差异可能来自物理运动，也可能来自通道宽度、Hanning smoothing 或速度参考系。观测设计和归档 QA 的记录越完整，后续科学解释越稳固。


### 9.23.7 本节结论

观测设计把科学问题转化为可测量的数据约束；归档数据再分析则反向检查已有数据是否满足这些约束。频率、带宽、通道宽度、积分时间、阵列配置、校准源和 QA 记录，都会进入后续校准、成像和科学量测量。许多看似“数据处理”的问题，其实是观测设计与科学目标不匹配的结果。

因此，实践训练不应只从已有 Measurement Set 开始，也应能够从科学目标推导观测需求，并从归档 metadata、weblog 和 pipeline 产品判断数据边界。这样得到的处理流程才不是命令集合，而是一条从科学问题到可复现结果的测量链。
